<center><font size="6" color="#2D6A4F"><b>simpleVIT+Linear decoder — OPE  Ablation</b></font></center>


**Environment Setup (Libraries & Installation)**

In [ ]:
!pip install -U albumentations[imgaug,extra,pytorch]
!pip install -U vit-pytorch
!pip install -U pytorch-lightning
!pip install -U torchmetrics


In [ ]:
import os
import json
import ast
import time
import random
import shutil
import logging
from datetime import datetime
from pathlib import Path
from typing import Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from scipy.stats import pearsonr
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as transforms

import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger

from torchmetrics import JaccardIndex
from torchmetrics.functional import jaccard_index, pearson_corrcoef

from vit_pytorch import SimpleViT

import albumentations as A
from albumentations.pytorch import ToTensorV2

from einops import rearrange


seed_value = 11
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
pl.seed_everything(seed_value)


import os, json, shutil, time, torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set VARIANT to one of the keys in VARIANTS, then run all. Repeat for each variant.

VARIANT = "baseline"   # "baseline" / "OPE_only"
VARIANTS = {
    "baseline": dict(use_orientation_pos_emb=False),
    "OPE_only": dict(use_orientation_pos_emb=True),
}
model_name = f"SimpleViT_linear_{VARIANT}"
base_log_dir = f"/content/drive/MyDrive/TranSpeckle/optuna_logs/{model_name}"
Path(base_log_dir).mkdir(parents=True, exist_ok=True)

**Data Augmentation**

In [ ]:
class AugmentedSpeckleDataset(Dataset):
    def __init__(self, x_tensor, y_tensor, transform):
        self.x_tensor = x_tensor
        self.y_tensor = y_tensor
        self.transform = transform

    def __len__(self):
        return self.x_tensor.size(0)

    def __getitem__(self, idx):
        x = self.x_tensor[idx]
        y = self.y_tensor[idx]

        x = x.permute(1, 2, 0).numpy()
        y = y.permute(1, 2, 0).numpy()

        augmented = self.transform(image=x, mask=y)
        x = augmented['image']
        y = augmented['mask']

        y = y.permute(2, 0, 1)

        return x.float(), y.float()

In [ ]:
class SpeckleDataModule(LightningDataModule):
    def __init__(
        self,
        train_data=None,
        val_data=None,
        test_data=None,
        batch_size: int = 16,
        num_workers: int = 4,
    ):
        super().__init__()
        self.train_data = train_data
        self.val_data = val_data
        self.test_data = test_data
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.train_transform = A.Compose([
            A.CropAndPad(percent=(-0.03, 0.0), border_mode=cv2.BORDER_CONSTANT, fill=0, keep_size=True, p=0.08),
            A.RandomBrightnessContrast(brightness_limit=0.08, p=0.05),
            A.Rotate(limit=10, p=0.07, border_mode=0,),
            A.ToTensorV2()
        ])

    def setup(self, stage: Optional[str] = None):
        if stage == 'fit' or stage is None:
            x_train, y_train = self.train_data
            x_val, y_val = self.val_data

            self.train_dataset = AugmentedSpeckleDataset(x_train, y_train, transform=self.train_transform)
            self.val_dataset = TensorDataset(x_val, y_val)

        if stage == 'test' or stage is None:
            x_test, y_test = self.test_data
            self.test_dataset = TensorDataset(x_test, y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

**Orientation-Aware Positional Embeddings**

In [ ]:
class OrientationPositionalEmbedding(nn.Module):
    def __init__(self, num_patches, emb_dim, grid_size=16, add_theta=True):
        super().__init__()
        self.emb_dim = emb_dim
        self.add_theta = add_theta
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, emb_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.2)

        if add_theta:
            self.theta_proj = nn.Linear(2, emb_dim)
            self.register_buffer("theta_encoding", self._create_theta_encoding(grid_size))

    def _create_theta_encoding(self, grid_size):
        encoding = torch.zeros(grid_size * grid_size, 2)
        center = grid_size / 2 - 0.5
        for i in range(grid_size):
            for j in range(grid_size):
                dx = j - center
                dy = i - center
                theta = math.atan2(dy, dx)
                idx = i * grid_size + j
                encoding[idx, 0] = math.sin(theta)
                encoding[idx, 1] = math.cos(theta)
        return encoding

    def forward(self, x):
        B, N, D = x.shape
        pos = self.pos_embed.repeat(B, 1, 1)
        if self.add_theta:
            theta_feats = self.theta_proj(self.theta_encoding)
            theta_feats = theta_feats.unsqueeze(0).repeat(B, 1, 1)
            pos = pos + theta_feats
        return x + pos


  **SimpleViT Segmentation Lightning Module (Linear Decoder + Orientation-Aware Positional Embeddings)**

In [ ]:
class LitSimpleViTWithDecoder(pl.LightningModule):
    def __init__(self, lr=1e-4, dim=512, depth=6, heads=8, mlp_dim=1024, patch_size=32,  use_orientation_pos_emb=False):
        super().__init__()
        self.save_hyperparameters()
        self.use_orientation_pos_emb = use_orientation_pos_emb


        self.model = SimpleViT(
            image_size=256,
            patch_size=patch_size,
            num_classes=2,
            dim=dim,
            depth=depth,
            heads=heads,
            mlp_dim=mlp_dim,
            channels=1
        )

        patch_area = patch_size * patch_size
        self.decoder = nn.Sequential(
            nn.Linear(dim, patch_area * 2),
        )

        self.lr = lr
        self.criterion = self.balanced_cross_entropy
        self.iou = JaccardIndex(task="binary", num_classes=2)

        self.patch_size = patch_size
        self.image_size = 256

        h = w = self.image_size // self.patch_size
        num_patches = h * w

        if self.use_orientation_pos_emb:
            self.pos_encoder = OrientationPositionalEmbedding(
                num_patches=num_patches,
                emb_dim=dim,
                grid_size=h,
                add_theta=True
            )
        else:
            self.pos_encoder = None

    def forward(self, img):
        B, C, H, W = img.shape
        x = self.model.to_patch_embedding(img)

        if self.pos_encoder is not None:
            x = self.pos_encoder(x)
        else:
            x = x + self.model.pos_embedding.to(x.device, dtype=x.dtype)

        x = self.model.transformer(x)
        x = self.decoder(x)

        p = self.patch_size
        h = w = self.image_size // p
        try:
            x = rearrange(x, 'b (h w) (c p1 p2) -> b c (h p1) (w p2)', h=h, w=w, p1=p, p2=p, c=2)
        except Exception as e:
            print("Rearranging failed:", e)
            print(f"x.shape = {x.shape}, h={h}, w={w}, p={p}")
            raise

        return x

    def balanced_cross_entropy(self, pred, target, eps=1e-6):
        pred_softmax = torch.softmax(pred, dim=1).clamp(min=eps, max=1 - eps)
        loss = - (target * torch.log(pred_softmax) + (1 - target) * torch.log(1 - pred_softmax))
        return loss.mean()

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = (logits[:, 1] > 0.5).long()
        targets = y[:, 1].long()

        iou = self.iou(preds, targets)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.log("train_iou", iou, prog_bar=True, on_step=False, on_epoch=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)

        preds = (logits[:, 1] > 0.5).long()
        targets = y[:, 1].long()

        iou = self.iou(preds, targets)

        self.log("val_loss", loss, on_step=False, on_epoch=True)
        self.log("val_iou", iou, prog_bar=True, on_step=False, on_epoch=True, logger=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        preds = (logits[:, 1] > 0.5).long()
        targets = y[:, 1].long()

        iou = self.iou(preds, targets)
        self.log("test_iou", iou)

        pcc = F.cosine_similarity(
            preds.flatten(start_dim=1).float(),
            targets.flatten(start_dim=1).float()
        ).mean()
        self.log("test_pcc", pcc)

    def configure_optimizers(self):
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)

            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="max",
                factor=0.5,
                patience=8,
                threshold=1e-4,
                cooldown=2,
                min_lr=1e-6,
            )

            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": "val_iou",
                    "interval": "epoch",
                    "frequency": 1,
                    "strict": True
                }
            }

**Evaluation**

In [ ]:
@torch.no_grad()
def predict_testset_torch(ckpt_path: str, x_test, batch_size: int = 16, device: str | None = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    x = torch.as_tensor(x_test).float()
    if x.ndim == 3:
        x = x.unsqueeze(1)
    dl = DataLoader(TensorDataset(x), batch_size=batch_size, shuffle=False)

    model = LitSimpleViTWithDecoder.load_from_checkpoint(ckpt_path).to(device).eval()

    preds = []
    for (xb,) in dl:
        xb = xb.to(device, non_blocking=True)
        probs = F.softmax(model(xb), dim=1).permute(0,2,3,1).cpu()
        preds.append(probs)
    return torch.cat(preds, dim=0).numpy()

In [ ]:
def evaluate_predictions_torchmetrics(y_pred: np.ndarray, y_true: np.ndarray, threshold: float = 0.5):

    if y_true.ndim == 4 and y_true.shape[-1] == 2:
        y_true_bin = np.argmax(y_true, axis=-1).astype(np.int32)
    else:
        y_true_bin = y_true.astype(np.int32)

    p1 = y_pred[..., 1]
    y_pred_bin = (p1 >= threshold).astype(np.int32)


    preds_t = torch.from_numpy(y_pred_bin)
    target_t = torch.from_numpy(y_true_bin)
    iou = jaccard_index(preds_t, target_t, task="binary").item()

    p1_t = torch.from_numpy(p1).float()
    gt_t = torch.from_numpy(y_true_bin).float()
    per_img_pcc = []
    for i in range(p1_t.shape[0]):
        per_img_pcc.append(pearson_corrcoef(p1_t[i].flatten(), gt_t[i].flatten()).item())
    pcc_mean = float(np.mean(per_img_pcc))

    print(f"IoU (mean over set): {iou:.3f}")
    print(f"PCC (Pearson, per-image mean): {pcc_mean:.3f}")

    return {
        "iou_mean": iou,
        "pcc_mean": pcc_mean,
        "pcc_per_image": np.array(per_img_pcc, dtype=np.float32),
    }

**Visualize model prediction vs. ground truth**



In [ ]:
def show_prediction_example_pt(x_test, y_true, y_pred, index=0, threshold: float = 0.5):

    def to_np(a):
        return a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a

    x = to_np(x_test)
    yt = to_np(y_true)
    yp = to_np(y_pred)

    xi, yi, pi = x[index], yt[index], yp[index]

    speckle = xi.squeeze()
    if speckle.ndim == 3:
        speckle = speckle[0] if speckle.shape[0] <= 3 else speckle[..., 0]

    if yi.ndim == 3 and yi.shape[0] == 2:
        gt_mask = np.argmax(yi, axis=0)
    elif yi.ndim == 3 and yi.shape[-1] == 2:
        gt_mask = np.argmax(yi, axis=-1)
    else:
        gt_mask = yi.squeeze().astype(np.int64)

    if pi.ndim == 3 and pi.shape[0] == 2:
        pred_mask = np.argmax(pi, axis=0)
    elif pi.ndim == 3 and pi.shape[-1] == 2:
        pred_mask = np.argmax(pi, axis=-1)
    else:
        pred_mask = (pi.squeeze() >= threshold).astype(np.int64)


    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(speckle, cmap='gray')
    plt.title("Speckle Input")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis('off')

    plt.suptitle(f"Example {index}", fontsize=14)
    plt.tight_layout()
    plt.show()


**Training Runner with DataModule & Checkpoint Resume**

In [ ]:
#Trains fixed runs using the best hyperparameters selected in the initial experiment,
#applied consistently across datasets of different sizes.

def train_fixed_run_dm(
    *,
    model_name: str,
    dataset_tag: str,
    model_class,
    hparams: dict,
    dm,
    base_log_dir: str = "/content/drive/MyDrive/TranSpeckle/optuna_logs",
    max_epochs: int = 350,
    patience: int = 35,
    resume: str = "fresh",
    init_ckpt_path: str | None = None
):
    import os, json, shutil, time, torch
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
    from pytorch_lightning.loggers import CSVLogger

    base_log_dir = os.path.join(base_log_dir, model_name)
    run_name = f"{model_name}_{dataset_tag}"
    run_dir  = os.path.join(base_log_dir, "fixed_runs", run_name)
    os.makedirs(run_dir, exist_ok=True)

    export_ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(export_ckpt_dir, exist_ok=True)

    tmp_ckpt_dir = os.path.join("/content/temp_checkpoints", run_name)
    os.makedirs(tmp_ckpt_dir, exist_ok=True)

    logger = CSVLogger(save_dir=run_dir, name="", version=".")
    lrmon  = LearningRateMonitor(logging_interval="epoch")

    ckpt_cb = ModelCheckpoint(
        dirpath=tmp_ckpt_dir,
        monitor="val_iou",
        mode="max",
        save_top_k=1,
        save_last=True,
        save_on_train_epoch_end=False
    )

    early = EarlyStopping(monitor="val_iou", mode="max", patience=patience, min_delta=1e-4)

    model = model_class(**hparams)

    resume_ckpt = None
    last_existing = os.path.join(export_ckpt_dir, f"last_{model_name}_{dataset_tag}.ckpt")

    if resume == "last" and os.path.exists(last_existing):
        resume_ckpt = last_existing
    elif resume == "from_path":
        assert init_ckpt_path and os.path.exists(init_ckpt_path), "init_ckpt_path not found."
        resume_ckpt = init_ckpt_path
    elif resume == "fresh":
        resume_ckpt = None

    if resume_ckpt:
        try:
            info = torch.load(resume_ckpt, map_location="cpu")
            print(f"[INFO] Resuming from: {resume_ckpt} | epoch in ckpt: {info.get('epoch','N/A')}")
        except Exception as e:
            print(f"[WARN] Could not read epoch from {resume_ckpt}: {e}")

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=[ckpt_cb, early, lrmon],
        logger=logger,
        accelerator="auto",
        enable_progress_bar=True,
        check_val_every_n_epoch=1,
        num_sanity_val_steps=0
    )


    print(f"[INFO] Run: {run_name} | Strategy: {resume}")
    t0 = time.time()
    trainer.fit(model, dm, ckpt_path=resume_ckpt)
    t1 = time.time()
    print(f"[INFO] Training time: {(t1 - t0)/60:.1f} min")

    def _copy(src, dst):
        if src and os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"[INFO] Saved: {dst}")
            return dst
        print(f"[WARN] Missing: {src}")
        return None

    last_out = _copy(ckpt_cb.last_model_path,
                     os.path.join(export_ckpt_dir, f"last_{model_name}_{dataset_tag}.ckpt"))
    best_out = _copy(ckpt_cb.best_model_path,
                     os.path.join(export_ckpt_dir, f"best_{model_name}_{dataset_tag}.ckpt"))

    cfg = {
        "run_name": run_name,
        "model_name": model_name,
        "dataset_tag": dataset_tag,
        "hparams": hparams,
        "patience": patience,
        "max_epochs": max_epochs,
        "resume": resume,
        "init_ckpt_path": init_ckpt_path,
        "last_ckpt": last_out,
        "best_ckpt": best_out,
        "metrics_csv": os.path.join(logger.log_dir, "metrics.csv"),
        "time_sec": t1 - t0
    }
    cfg_path = os.path.join(run_dir, "run_config.json")
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"[INFO] Wrote config: {cfg_path}")

    def _as_float(x):
        try:
            return float(x.item()) if hasattr(x, "item") else float(x)
        except Exception:
            return None

    final_val_iou = _as_float(trainer.callback_metrics.get("val_iou"))
    best_val_iou  = _as_float(ckpt_cb.best_model_score)
    best_path     = ckpt_cb.best_model_path

    if final_val_iou is not None:
        print(f"[RESULT] Final val_iou: {final_val_iou:.6f}")
    else:
        print("[RESULT] Final val_iou: N/A (no metric logged)")

    if best_val_iou is not None and best_path:
        print(f"[RESULT] Best val_iou : {best_val_iou:.6f}  | best.ckpt: {best_path}")
    else:
        print("[RESULT] Best val_iou : N/A")

    return {
        "run_dir": run_dir,
        "last_ckpt": last_out,
        "best_ckpt": best_out,
        "metrics_csv": cfg["metrics_csv"],
        "config": cfg_path,
        "final_val_iou": float(trainer.callback_metrics["val_iou"].item()) if "val_iou" in trainer.callback_metrics else None

    }

**Hyperparameters with Variant**

In [ ]:
# Hyperparameters from the Optuna search in the main notebook (02_simpleViT_linear_decoder)

hparams = dict(dim=512, heads=16, depth=4, mlp_dim = 2048, lr=7.495007493463695e-05, patch_size=32, **VARIANTS[VARIANT])

<center><font size="5" color="#2D6A4F"><b>Model2400</b></font></center>

**Load Processed Data + Lightning DataModule**

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/2400/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Train Variant**

In [ ]:
out = train_fixed_run_dm(model_name=model_name, dataset_tag="2400", model_class=LitSimpleViTWithDecoder,
                         hparams=hparams, dm=dm, base_log_dir=base_log_dir,
                         max_epochs=500, resume="fresh")

<center><font size="5" color="#2D6A4F"><b>Model1600_4</b></font></center>

**Load Processed Data + Lightning DataModule**

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/1600_4/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_1600_4 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Train Variant**

In [ ]:
out = train_fixed_run_dm(model_name=model_name, dataset_tag="1600_4", model_class=LitSimpleViTWithDecoder,
                         hparams=hparams, dm=dm_1600_4, base_log_dir=base_log_dir,
                         max_epochs=500, resume="fresh")

<center><font size="5" color="#2D6A4F"><b>Model1600_2</b></font></center>

**Load Processed Data + Lightning DataModule**

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/1600_2/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_1600_2 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Train Variant**

In [ ]:
out = train_fixed_run_dm(model_name=model_name, dataset_tag="1600_2", model_class=LitSimpleViTWithDecoder,
                         hparams=hparams, dm=dm_1600_2, base_log_dir=base_log_dir,
                         max_epochs=500, resume="fresh")

<center><font size="5" color="#2D6A4F"><b>Model800_4</b></font></center>

**Load Processed Data + Lightning DataModule**

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/800_4/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_800_4 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Train Variant**

In [ ]:
out = train_fixed_run_dm(model_name=model_name, dataset_tag="800_4", model_class=LitSimpleViTWithDecoder,
                         hparams=hparams, dm=dm_800_4, base_log_dir=base_log_dir,
                         max_epochs=500, resume="fresh")

<center><font size="5" color="#2D6A4F"><b>Model800_2</b></font></center>

**Load Processed Data + Lightning DataModule**

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/800_2/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_800_2 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Train Variant**

In [ ]:
out = train_fixed_run_dm(model_name=model_name, dataset_tag="800_2", model_class=LitSimpleViTWithDecoder,
                         hparams=hparams, dm=dm_800_2, base_log_dir=base_log_dir,
                         max_epochs=500, resume="fresh")

<center><font size="5" color="#2D6A4F"><b>Model800_1</b></font></center>

**Load Processed Data + Lightning DataModule**

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/800_1/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_800_1 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Train Variant**

In [ ]:
out = train_fixed_run_dm(model_name=model_name, dataset_tag="800_1", model_class=LitSimpleViTWithDecoder,
                         hparams=hparams, dm=dm_800_1, base_log_dir=base_log_dir,
                         max_epochs=500, resume="fresh")



---



**Evaluate All Variants (run after all variants are trained)**

In [ ]:
import pandas as pd
ROOT = "/content/drive/MyDrive/TranSpeckle/optuna_logs"
RESULTS_CSV = "/content/drive/MyDrive/TranSpeckle/ablation_results_SimpleViT.csv"

EVAL_PLAN = {
    "2400":   ["group1"],
    "1600_4": ["group1"],
    "1600_2": ["group1"],
    "800_4":  ["group1"],
    "800_2":  ["group1"],
    "800_1":  ["group1"],
}

MODELS_TO_EVAL = ["SimpleViT_linear_baseline", "SimpleViT_linear_OPE_only"]

def ckpt_for(name, tag):
    return f"{ROOT}/{name}/{name}/fixed_runs/{name}_{tag}/checkpoints/last_{name}_{tag}.ckpt", "last"

def load_test(group):
    suffix = group.replace("group", "")
    d = torch.load(f"/content/drive/MyDrive/TranSpeckle/cache/{group}/processed.pt")
    return d[f"x_test{suffix}"], d[f"y_test{suffix}"]

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0, 2, 3, 1))
    return y_np

rows = []
test_cache = {}
for name in MODELS_TO_EVAL:
    for tag, groups in EVAL_PLAN.items():
        ckpt_path, ckpt_kind = ckpt_for(name, tag)
        if not os.path.exists(ckpt_path):
            print(f"[SKIP] {name} | {tag}: no checkpoint")
            continue
        for group in groups:
            if group not in test_cache:
                test_cache[group] = load_test(group)
            x_t, y_t = test_cache[group]
            print(f"\n=== {name} | train {tag} | test {group} ===")
            y_pred = predict_testset_torch(ckpt_path=ckpt_path, x_test=x_t, batch_size=100)
            res = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=_to_numpy_gt(y_t), threshold=0.5)
            rows.append(dict(model=name, train_set=tag, test_set=group, ckpt=ckpt_kind,
                             iou=res["iou_mean"], pcc=res["pcc_mean"]))

new = pd.DataFrame(rows)
if os.path.exists(RESULTS_CSV) and len(new):
    old = pd.read_csv(RESULTS_CSV)
    keys = ["model", "train_set", "test_set"]
    old = old[~old.set_index(keys).index.isin(new.set_index(keys).index)]   # replace re-evaluated rows
    new = pd.concat([old, new], ignore_index=True)
if len(new):
    new.to_csv(RESULTS_CSV, index=False)
    print(f"\n[INFO] Saved {len(rows)} results to {RESULTS_CSV}")

**Ablation Summary**

In [ ]:
import pandas as pd
RESULTS_CSV = "/content/drive/MyDrive/TranSpeckle/ablation_results_SimpleViT.csv"

BASE = "SimpleViT_linear_baseline"
OPE  = "SimpleViT_linear_OPE_only"
ORDER = ["2400", "1600_4", "1600_2", "800_4", "800_2", "800_1"]

# Paper numbers for "Simple ViT + Linear Decoder" (Tables 2 & 3, group1), same order as ORDER
PAPER = {"iou": [0.359, 0.219, 0.293, 0.184, 0.235, 0.213],
         "pcc": [0.559, 0.502, 0.530, 0.464, 0.484, 0.466]}

df = pd.read_csv(RESULTS_CSV)
df = df[df["test_set"] == "group1"]

missing = [m for m in [BASE, OPE] if m not in df["model"].unique()]
if missing:
    print("Not all variants have been trained yet. Missing:", missing)
else:
    for metric in ["iou", "pcc"]:
        t = df.pivot_table(index="train_set", columns="model", values=metric).reindex(ORDER)

        # 1. Ablation: OPE vs. the model's built-in positional embedding
        abl = pd.DataFrame(index=t.index)
        abl["baseline"] = t[BASE]
        abl["OPE"] = t[OPE]
        abl["Δ OPE"] = t[OPE] - t[BASE]
        abl.loc["mean"] = abl.mean()

        # 2. Reproduction check: re-trained OPE model vs. the numbers reported in the paper
        rep = pd.DataFrame(index=t.index)
        rep["paper model"] = PAPER[metric]
        rep["OPE (re-trained)"] = t[OPE]
        rep["Δ"] = rep["OPE (re-trained)"] - rep["paper model"]

        print(f"\n===== {metric.upper()}: ablation (Δ vs. baseline) =====")
        display(abl.round(4))
        print(f"\n----- {metric.upper()}: reproduction check vs. paper -----")
        display(rep.round(4))